This is our notebook

load data

In [2]:
import pandas as pd

df = pd.read_csv("training_v2.csv")
df.head()

,encounter_id,patient_id,hospital_id,hospital_death,age,bmi,elective_surgery,ethnicity,gender,height,...,aids,cirrhosis,diabetes_mellitus,hepatic_failure,immunosuppression,leukemia,lymphoma,solid_tumor_with_metastasis,apache_3j_bodysystem,apache_2_bodysystem
0,66154,25312,118,0,68.0,22.73,0,Caucasian,M,180.3,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Sepsis,Cardiovascular
1,114252,59342,81,0,77.0,27.42,0,Caucasian,F,160.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Respiratory,Respiratory
2,119783,50777,118,0,25.0,31.95,0,Caucasian,F,172.7,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Metabolic,Metabolic
3,79267,46918,118,0,81.0,22.64,1,Caucasian,F,165.1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Cardiovascular,Cardiovascular
4,92056,34377,33,0,19.0,NaN,0,Caucasian,M,188.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Trauma,Trauma


Unlabeled dataset (dataset.csv)

This is typically:
Use this after training to: Make predictions, Simulate “real-world” unseen data, this will be our analysis. 

clean data, encode categorical variables and define features and target

In [ ]:
# Drop ID-like columns (not useful for prediction)
df = df.drop(columns=["encounter_id", "patient_id"], errors="ignore")

# Handle missing values
# numeric fill
num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# categorical fill
cat_cols = df.select_dtypes(exclude=["number"]).columns
df[cat_cols] = df[cat_cols].fillna("Unknown")

df = pd.get_dummies(df, drop_first=True)

X = df.drop("hospital_death", axis=1)
y = df["hospital_death"]

baselime nodel will be a logistic regression because the model has to be basic not the data apparently...

Logistic regression was used as a linear probabilistic classifier representing a simple decision boundary.\

Logistic regression convergence depends on feature scaling and solver choice. High-dimensional sparse feature spaces can slow optimization and require more robust solvers such as SAGA.

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(X, y)

baseline_preds = baseline_model.predict(X)
baseline_probs = baseline_model.predict_proba(X)[:, 1]

the actual model basic version, im also going to do at least one improved version where we can try with more dataparsing or specific settings

A support vector machine with an RBF kernel was used to construct a nonlinear maximum-margin classifier.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

basic_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True))
])

basic_svm.fit(X, y)

basic_svm_preds = basic_svm.predict(X)
basic_svm_probs = basic_svm.predict_proba(X)[:, 1]

adding hyper parameters

The effect of hyperparameters (C and gamma) was evaluated to demonstrate changes in model complexity and overfitting behavior.

In [ ]:
svm_tuned = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=10, gamma=0.01, probability=True))
])

svm_tuned.fit(X, y)

svm_tuned_preds = svm_tuned.predict(X)
svm_tuned_probs = svm_tuned.predict_proba(X)[:, 1]

adding another svm that has less feature but still not hyperparamenters, Reducing the feature space allowed us to analyze the effect of dimensionality on SVM performance

In [ ]:
selected_features = [
    "age",
    "bmi",
    "heart_rate_apache",
    "gcs_motor_apache",
    "d1_bun_max",
    "d1_creatinine_max"
]

X_reduced = X[selected_features]

svm_reduced = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1, gamma="scale", probability=True))
])

svm_reduced.fit(X_reduced, y)

svm_reduced_preds = svm_reduced.predict(X_reduced)
svm_reduced_probs = svm_reduced.predict_proba(X_reduced)[:, 1]